In [38]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [39]:
# turning big numbers into 0. numbers, for model to understand better
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [41]:
# splitting the training data into training and validation sets
rng = np.random.default_rng(42)
idx = rng.permutation(len(x_train))

x_val, y_val = x_train[idx[:10000]], y_train[idx[:10000]]
x_train, y_train = x_train[idx[10000:]], y_train[idx[10000:]]

print(x_train.shape, y_train.shape)

(40000, 28, 28) (40000,)


In [42]:
# reshaping the data to be 2D array for the model
# x_train_flat = x_train.reshape(-1, 28*28)
# x_val_flat = x_val.reshape(-1, 28*28)
# x_test_flat = x_test.reshape(-1, 28*28)

# converting the labels to int64 for compatibility with TensorFlow
y_train = y_train.astype('int64')
y_val = y_val.astype('int64')
y_test = y_test.astype('int64')

In [43]:
tf.random.set_seed(42)
# initializing weights and biases
# W1 = tf.Variable(tf.random.normal((784, 128)) * (2 / 784) ** 0.5)
#128 zeros for each of the 128 neurons in the hidden layer
# b1 = tf.Variable(tf.zeros(128))
# W2 = tf.Variable(tf.random.normal((128, 10)) * (2 / 128) ** 0.5)
# b2 = tf.Variable(tf.zeros(10))
# params = [W1, b1, W2, b2]

# keras creates weights inside the layers, so we no longer write tf.Variable ourselves.
model = keras.Sequential([
    keras.Input(shape=(28,28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10),
])
print(model.summary())



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

None


In [48]:
# PART 3: THE NETWORK'S BUILDING BLOCKS
# Three functions we will use again and again:
#   forward     -> runs images through the network and gets 10 scores per image
#   loss_fn     -> measures how wrong those scores are (one number, lower = better)
#   accuracy_fn -> measures what fraction of images the network gets right

model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
# PART 4: UNDERSTANDING CHECK
# Goal: calculate the gradient of W2 two ways (TensorFlow automatically, and by hand
# with a formula) and check that both give the same answer.
# Gradient = for every weight: "if I nudge it up a tiny bit, does the loss go up or down, and how much?"


In [ ]:
history = model.fit(x_train, y_train, batch_size=64, epochs=15, validation_data=(x_val, y_val))
print(history.history.keys())

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 673us/step - accuracy: 0.9750 - loss: 0.0833 - val_accuracy: 0.9680 - val_loss: 0.1047
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 477us/step - accuracy: 0.9819 - loss: 0.0633 - val_accuracy: 0.9695 - val_loss: 0.0987
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - accuracy: 0.9865 - loss: 0.0499 - val_accuracy: 0.9720 - val_loss: 0.0931
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 453us/step - accuracy: 0.9897 - loss: 0.0396 - val_accuracy: 0.9727 - val_loss: 0.0892
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 477us/step - accuracy: 0.9932 - loss: 0.0309 - val_accuracy: 0.9728 - val_loss: 0.0881
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 530us/step - accuracy: 0.9954 - loss: 0.0246 - val_accuracy: 0.9742 - val_loss: 0.0867
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.9967 - loss: 0.0197 - val_accuracy: 0.9744 - val_loss: 0.0860
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 515us/step - accuracy: 0.9977 - loss: 0.0158 - 

In [50]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 429us/step - accuracy: 0.9761 - loss: 0.0893
Test accuracy: 0.9761, Test loss: 0.0893
